# 06 - CI Gate For Data And AI Changes

The reusable `cicd_policy_gate` package maps a pull request's change set
(SQL models, export jobs, AI workflows — each carrying a canonical
`scenario_key`) onto validate/authorize calls and turns the typed answers into
pass / needs_controls / fail gates with reviewable reason codes.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None):
    ref = {"database": "acmecloud_demo", "schema": "public", "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


In [ ]:
from cicd_policy_gate.gate import evaluate_changes, load_changes

change_set = load_changes()
summary = evaluate_changes(client, change_set, strict=True)
print(f"pass={summary.passed} needs_controls={summary.needs_controls} fail={summary.failed}")
print(f"release_allowed: {summary.release_allowed}")


In [ ]:
for result in summary.results:
    print(f"{result.change_id}: {result.gate} ({result.decision})")
    if result.reason_codes:
        print(f"  reason_codes: {', '.join(result.reason_codes)}")
    if result.required_controls:
        print(f"  controls: {'; '.join(result.required_controls)}")


In CI, run `scripts/run_cicd_policy_gate.sh --strict` — the exit code blocks the
merge when denied changes are present. See `docs/ci-cd-policy-gate.md` for the
GitHub Actions shape.
